# Image Captioning Aksara Lontara
## Fine-Tuning BLIP | Tesis S2

Notebook ini menjalankan **seluruh pipeline** dalam satu proses:
1. Mount Google Drive & Install Library
2. Persiapan Dataset → Training → Evaluasi → Inference

> **Pastikan runtime menggunakan GPU** (Runtime → Change runtime type → GPU)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SETUP: Mount Google Drive & Install Library
# ═══════════════════════════════════════════════════════════════
import os
try:
    os.chdir('/content')
except OSError:
    pass

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/ImageCaptioning'
LOCAL_DIR = '/content/ImageCaptioning'

# Copy dataset dari Drive ke local (lebih stabil & cepat)
assert os.path.exists(DRIVE_DIR), f'ERROR: {DRIVE_DIR} tidak ditemukan!'
!rm -rf {LOCAL_DIR}
!cp -r {DRIVE_DIR} {LOCAL_DIR}

os.chdir(LOCAL_DIR)
print(f'\u2705 Working directory: {os.getcwd()}')
print(f'\u2705 Isi folder: {os.listdir(".")}')

# Install library
!pip install -q transformers Pillow nltk rouge-score tqdm

import torch
print(f'\u2705 PyTorch: {torch.__version__}')
print(f'\u2705 CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'\u2705 GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PIPELINE LENGKAP: Prepare → Train → Evaluate → Inference
# ═══════════════════════════════════════════════════════════════

import os
import json
import random
import shutil
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from tqdm import tqdm
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
import nltk
import warnings
warnings.filterwarnings('ignore')

nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('omw-1.4', quiet=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')


# ═══════════════════════════════════════════════════════════════
# TAHAP 1: PERSIAPAN DATASET
# ═══════════════════════════════════════════════════════════════
print('\n' + '=' * 60)
print('  TAHAP 1: PERSIAPAN DATASET')
print('=' * 60)

RAW_IMAGE_DIR = 'dataset/images'
CAPTION_FILE  = 'dataset/captions.json'
OUTPUT_DIR    = 'dataset/processed'
IMAGE_SIZE    = 384
RANDOM_SEED   = 42

KARAKTER_DASAR = [
    'a', 'ba', 'ca', 'da', 'ga',
    'ha', 'ja', 'ka', 'la', 'ma',
    'na', 'nga', 'nya', 'pa', 'ra',
    'sa', 'ta', 'wa', 'ya'
]
VOKAL = ['a', 'i', 'u', 'e', 'o']

# Buat mapping ltr-XX → karakter
mapping = {}
idx = 1
for dasar in KARAKTER_DASAR:
    for vokal in VOKAL:
        ltr_name = f'ltr-{idx:02d}'
        mapping[ltr_name] = {'karakter': dasar, 'vokal': vokal}
        idx += 1

# Buat folder output
for split in ['train', 'val', 'test']:
    os.makedirs(f'{OUTPUT_DIR}/{split}/images', exist_ok=True)

# Load captions.json
with open(CAPTION_FILE, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)
print(f'Caption dimuat: {len(raw_data)} entri')

# Flatten: 1 gambar x 5 caption = 5 pasang
flat_data = []
for item in raw_data:
    ltr_name = item['images']
    img_filename = f'{ltr_name}.png'
    info = mapping.get(ltr_name, {})
    for caption in item['captions']:
        flat_data.append({
            'image': img_filename,
            'caption': caption,
            'karakter': info.get('karakter', ''),
            'vokal': info.get('vokal', ''),
        })
print(f'Total pasang: {len(flat_data)}')

# Split dataset
random.seed(RANDOM_SEED)
random.shuffle(flat_data)
n = len(flat_data)
n_train = int(n * 0.8)
n_val = int(n * 0.1)
train_data = flat_data[:n_train]
val_data = flat_data[n_train:n_train + n_val]
test_data = flat_data[n_train + n_val:]
print(f'Split: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}')

# Simpan JSON per split
for name, data in [('train', train_data), ('val', val_data), ('test', test_data)]:
    path = f'{OUTPUT_DIR}/{name}/captions_{name}.json'
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

# Resize & copy gambar per split
for name, data in [('train', train_data), ('val', val_data), ('test', test_data)]:
    processed = set()
    for item in data:
        img = item['image']
        if img in processed:
            continue
        processed.add(img)
        src = os.path.join(RAW_IMAGE_DIR, img)
        dst = os.path.join(OUTPUT_DIR, name, 'images', img)
        if os.path.exists(src):
            im = Image.open(src).convert('RGB')
            im.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS).save(dst)
    print(f'  {name}: {len(processed)} gambar diproses')

print('\u2705 Persiapan dataset selesai!')


# ═══════════════════════════════════════════════════════════════
# TAHAP 2: FINE-TUNING BLIP
# ═══════════════════════════════════════════════════════════════
print('\n' + '=' * 60)
print('  TAHAP 2: FINE-TUNING BLIP')
print('=' * 60)

MODEL_NAME = 'Salesforce/blip-image-captioning-base'
MODEL_OUTPUT = 'model/blip_lontara'
NUM_EPOCHS = 20
BATCH_SIZE = 8
LR = 2e-5
MAX_LENGTH = 64

os.makedirs(MODEL_OUTPUT, exist_ok=True)


class AksaraDataset(Dataset):
    def __init__(self, json_file, image_dir, processor):
        with open(json_file, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        self.image_dir = image_dir
        self.processor = processor
        print(f'  Dataset: {len(self.data)} pasang dari {json_file}')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img_path = os.path.join(self.image_dir, os.path.basename(item['image']))
        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            image = Image.new('RGB', (384, 384), (255, 255, 255))

        encoding = self.processor(
            images=image, text=item['caption'],
            padding='max_length', truncation=True,
            max_length=MAX_LENGTH, return_tensors='pt'
        )
        input_ids = encoding['input_ids'].squeeze()
        attention_mask = encoding['attention_mask'].squeeze()
        pixel_values = encoding['pixel_values'].squeeze()
        labels = input_ids.clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'input_ids': input_ids,
                'attention_mask': attention_mask, 'labels': labels}


# Load model
print('Memuat model BLIP...')
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
print(f'Model dimuat: {sum(p.numel() for p in model.parameters()):,} parameter')

# DataLoader
train_ds = AksaraDataset(f'{OUTPUT_DIR}/train/captions_train.json', f'{OUTPUT_DIR}/train/images', processor)
val_ds = AksaraDataset(f'{OUTPUT_DIR}/val/captions_val.json', f'{OUTPUT_DIR}/val/images', processor)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Optimizer & Scheduler
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=total_steps)
print(f'Total steps: {total_steps}')

# Training loop
history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    # Train
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [Train]', leave=False):
        pv = batch['pixel_values'].to(DEVICE)
        ids = batch['input_ids'].to(DEVICE)
        am = batch['attention_mask'].to(DEVICE)
        lb = batch['labels'].to(DEVICE)
        outputs = model(pixel_values=pv, input_ids=ids, attention_mask=am, labels=lb)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [Val]', leave=False):
            pv = batch['pixel_values'].to(DEVICE)
            ids = batch['input_ids'].to(DEVICE)
            am = batch['attention_mask'].to(DEVICE)
            lb = batch['labels'].to(DEVICE)
            outputs = model(pixel_values=pv, input_ids=ids, attention_mask=am, labels=lb)
            total_loss += outputs.loss.item()
    val_loss = total_loss / len(val_loader)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}')

    # Save best
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_dir = f'{MODEL_OUTPUT}/best_model'
        model.save_pretrained(best_dir)
        processor.save_pretrained(best_dir)
        print(f'  \u2b50 Best model saved! Val Loss: {best_val_loss:.4f}')

    # Checkpoint setiap 5 epoch
    if epoch % 5 == 0:
        ckpt_dir = f'{MODEL_OUTPUT}/checkpoint_epoch_{epoch}'
        os.makedirs(ckpt_dir, exist_ok=True)
        model.save_pretrained(ckpt_dir)
        processor.save_pretrained(ckpt_dir)

# Simpan history
with open(f'{MODEL_OUTPUT}/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\n\u2705 Training selesai! Best Val Loss: {best_val_loss:.4f}')

# Plot
plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], label='Train Loss', marker='o')
plt.plot(history['val_loss'], label='Val Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training History')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{MODEL_OUTPUT}/training_plot.png', dpi=150)
plt.show()


# ═══════════════════════════════════════════════════════════════
# TAHAP 3: EVALUASI
# ═══════════════════════════════════════════════════════════════
print('\n' + '=' * 60)
print('  TAHAP 3: EVALUASI MODEL')
print('=' * 60)

# Load best model
best_dir = f'{MODEL_OUTPUT}/best_model'
eval_processor = BlipProcessor.from_pretrained(best_dir)
eval_model = BlipForConditionalGeneration.from_pretrained(best_dir).to(DEVICE)
eval_model.eval()
print('Model evaluasi dimuat.')

# Load test data
with open(f'{OUTPUT_DIR}/test/captions_test.json', 'r', encoding='utf-8') as f:
    test_items = json.load(f)
print(f'Data test: {len(test_items)} entri')

# Kelompokkan per gambar
gambar_dict = {}
for item in test_items:
    key = item['image']
    if key not in gambar_dict:
        gambar_dict[key] = {'captions': [], 'karakter': item.get('karakter', ''), 'vokal': item.get('vokal', '')}
    gambar_dict[key]['captions'].append(item['caption'])

# Generate & evaluate
referensi_list = []
hipotesis_list = []
hasil_detail = []

for image_key, info in tqdm(gambar_dict.items(), desc='Evaluasi'):
    img_path = os.path.join(f'{OUTPUT_DIR}/test/images', os.path.basename(image_key))
    try:
        image = Image.open(img_path).convert('RGB')
    except FileNotFoundError:
        continue
    inputs = eval_processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        output = eval_model.generate(**inputs, max_length=64, num_beams=4, early_stopping=True)
    pred = eval_processor.decode(output[0], skip_special_tokens=True)

    refs = [cap.lower().split() for cap in info['captions']]
    hyp = pred.lower().split()
    referensi_list.append(refs)
    hipotesis_list.append(hyp)
    hasil_detail.append({
        'image': image_key, 'karakter': info['karakter'], 'vokal': info['vokal'],
        'caption_prediksi': pred, 'captions_referensi': info['captions']
    })

# Hitung metrik
smoother = SmoothingFunction().method1
bleu1 = corpus_bleu(referensi_list, hipotesis_list, weights=(1,0,0,0), smoothing_function=smoother)
bleu2 = corpus_bleu(referensi_list, hipotesis_list, weights=(0.5,0.5,0,0), smoothing_function=smoother)
bleu3 = corpus_bleu(referensi_list, hipotesis_list, weights=(0.33,0.33,0.33,0), smoothing_function=smoother)
bleu4 = corpus_bleu(referensi_list, hipotesis_list, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoother)

meteor_scores_list = []
for refs, hyp in zip(referensi_list, hipotesis_list):
    ref_strs = [' '.join(r) for r in refs]
    meteor_scores_list.append(meteor_score(ref_strs, ' '.join(hyp)))
meteor_avg = np.mean(meteor_scores_list)

rouge_sc = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
rouge_scores_list = []
for refs, hyp in zip(referensi_list, hipotesis_list):
    hyp_str = ' '.join(hyp)
    scores = [rouge_sc.score(' '.join(r), hyp_str)['rougeL'].fmeasure for r in refs]
    rouge_scores_list.append(max(scores))
rouge_avg = np.mean(rouge_scores_list)

print('\n' + '=' * 60)
print('  HASIL EVALUASI')
print('=' * 60)
metrik = {
    'BLEU-1': bleu1*100, 'BLEU-2': bleu2*100,
    'BLEU-3': bleu3*100, 'BLEU-4': bleu4*100,
    'METEOR': meteor_avg*100, 'ROUGE-L': rouge_avg*100
}
for k, v in metrik.items():
    print(f'  {k:<10}: {v:>6.2f}%')

# Simpan hasil
os.makedirs('hasil_evaluasi', exist_ok=True)
hasil_final = {'metrik': {k: round(v, 2) for k, v in metrik.items()}, 'detail': hasil_detail}
with open('hasil_evaluasi/hasil_evaluasi.json', 'w', encoding='utf-8') as f:
    json.dump(hasil_final, f, ensure_ascii=False, indent=2)
print('\u2705 Hasil evaluasi disimpan: hasil_evaluasi/hasil_evaluasi.json')


# ═══════════════════════════════════════════════════════════════
# TAHAP 4: INFERENCE & VISUALISASI
# ═══════════════════════════════════════════════════════════════
print('\n' + '=' * 60)
print('  TAHAP 4: INFERENCE & VISUALISASI')
print('=' * 60)

test_folder = f'{OUTPUT_DIR}/test/images'
test_images = sorted([f for f in os.listdir(test_folder) if f.endswith('.png')])[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

inference_results = []
for idx, img_name in enumerate(test_images):
    img_path = os.path.join(test_folder, img_name)
    image = Image.open(img_path).convert('RGB')
    inputs = eval_processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        output = eval_model.generate(**inputs, max_length=64, num_beams=4, early_stopping=True, no_repeat_ngram_size=2)
    caption = eval_processor.decode(output[0], skip_special_tokens=True)
    inference_results.append({'gambar': img_name, 'caption': caption})
    print(f'  {img_name} \u2192 {caption}')

    axes[idx].imshow(image)
    axes[idx].set_title(caption, fontsize=9, wrap=True)
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('hasil_evaluasi/visualisasi_prediksi.png', dpi=150, bbox_inches='tight')
plt.show()

# Simpan hasil inference
with open('hasil_evaluasi/hasil_inference.json', 'w', encoding='utf-8') as f:
    json.dump(inference_results, f, ensure_ascii=False, indent=2)


# ═══════════════════════════════════════════════════════════════
# TAHAP 5: COPY HASIL KE GOOGLE DRIVE
# ═══════════════════════════════════════════════════════════════
print('\n' + '=' * 60)
print('  TAHAP 5: MENYIMPAN HASIL KE GOOGLE DRIVE')
print('=' * 60)

DRIVE_DIR = '/content/drive/MyDrive/ImageCaptioning'

# Copy model & hasil evaluasi ke Drive
for folder in ['model', 'hasil_evaluasi']:
    src = os.path.join(LOCAL_DIR, folder)
    dst = os.path.join(DRIVE_DIR, folder)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'  \u2705 {folder}/ → Drive')

print('\n' + '=' * 60)
print('  \u2705 SELURUH PIPELINE SELESAI!')
print(f'  Model terbaik : {DRIVE_DIR}/model/blip_lontara/best_model')
print(f'  Hasil evaluasi: {DRIVE_DIR}/hasil_evaluasi/')
print('=' * 60)